In [10]:
# ==============================================================================
# WASTE MANAGEMENT ML PROJECT - FEATURE ENGINEERING
# ==============================================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler
import warnings
warnings.filterwarnings('ignore')

# Load data

data_path = 'C:/Users/user/Desktop/DecisionTree/data/processed/features.csv'

df = pd.read_csv(data_path)

print(f"Dataset shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")

Dataset shape: (107949, 47)
Columns: ['collection_id', 'bin_id', 'collection_date', 'collection_time', 'day_of_week', 'is_holiday', 'holiday_name', 'is_festival_period', 'festival_name', 'days_since_last_collection', 'fill_level_percent', 'fill_level_category', 'waste_weight_kg', 'waste_type_primary', 'waste_composition_json', 'odor_complaint', 'overflow_reported', 'collector_notes', 'temperature_c', 'rainfall_mm', 'humidity_percent', 'location_type', 'district', 'capacity_liters', 'bin_material', 'has_lid', 'nearby_population', 'distance_to_depot_km', 'road_accessibility', 'installation_date', 'latitude', 'longitude', 'month', 'week_of_year', 'is_weekend', 'is_month_start', 'is_month_end', 'fill_rate_7day_avg', 'fill_rate_7day_std', 'weight_7day_avg', 'prev_fill_level', 'prev_days_since_collection', 'bin_age_days', 'fill_rate_per_day', 'overflow_count_30day', 'organic_percent', 'recyclable_percent']


In [11]:
# ==============================================================================
# 2. ADD MISSING TEMPORAL FEATURES
# ==============================================================================

# Convert to datetime if needed
df['collection_date'] = pd.to_datetime(df['collection_date'])

# Season feature (Ghana has wet/dry seasons)
def get_season(month):
    """Ghana seasons: Dry (Nov-Mar), Wet (Apr-Oct)"""
    if month in [11, 12, 1, 2, 3]:
        return 'dry'
    else:
        return 'wet'

df['season'] = df['month'].apply(get_season)
print("✅ Added 'season' feature (dry/wet)")

# Quarter
df['quarter'] = df['month'].apply(lambda x: (x - 1) // 3 + 1)
print("✅ Added 'quarter' feature")

# Day of month
df['day_of_month'] = df['collection_date'].dt.day
print("✅ Added 'day_of_month' feature")

# Is beginning of month (days 1-7)
df['is_early_month'] = (df['day_of_month'] <= 7).astype(int)
print("✅ Added 'is_early_month' feature")

# Is end of month (days 25+)
df['is_late_month'] = (df['day_of_month'] >= 25).astype(int)
print("✅ Added 'is_late_month' feature")

print(f"\nNew shape: {df.shape}")

✅ Added 'season' feature (dry/wet)
✅ Added 'quarter' feature
✅ Added 'day_of_month' feature
✅ Added 'is_early_month' feature
✅ Added 'is_late_month' feature

New shape: (107949, 52)


In [12]:
# ==============================================================================
# 3. CREATE INTERACTION FEATURES
# ==============================================================================

print("\n🔗 Creating Interaction Features...")

# Location × Time interactions
df['location_weekend'] = df['location_type'] + '_' + df['is_weekend'].astype(str)
print("✅ Added 'location_weekend' interaction")

# Fill rate × Days since collection
df['fill_momentum'] = df['fill_rate_per_day'] * df['days_since_last_collection']
print("✅ Added 'fill_momentum' feature")

# Capacity utilization trend
df['capacity_fill_ratio'] = df['fill_level_percent'] / (df['capacity_liters'] / 1000)
print("✅ Added 'capacity_fill_ratio' feature")

# Weather impact score
df['weather_score'] = (df['temperature_c'] / 35) - (df['rainfall_mm'] / 50) + (df['humidity_percent'] / 100)
print("✅ Added 'weather_score' feature")

# Population density impact
df['population_fill_ratio'] = df['fill_level_percent'] / (df['nearby_population'] / 1000)
print("✅ Added 'population_fill_ratio' feature")

# Risk score (composite)
df['overflow_risk_score'] = (
    df['fill_rate_7day_avg'] * 0.4 +
    df['days_since_last_collection'] * 5 +
    df['overflow_count_30day'] * 10
)
print("✅ Added 'overflow_risk_score' feature")

print(f"\nNew shape: {df.shape}")


🔗 Creating Interaction Features...
✅ Added 'location_weekend' interaction
✅ Added 'fill_momentum' feature
✅ Added 'capacity_fill_ratio' feature
✅ Added 'weather_score' feature
✅ Added 'population_fill_ratio' feature
✅ Added 'overflow_risk_score' feature

New shape: (107949, 58)


In [13]:
# ==============================================================================
# 4. HANDLE MISSING VALUES
# ==============================================================================

print("\n🔧 Handling Missing Values...")
print("\nMissing values before:")
missing = df.isnull().sum()
print(missing[missing > 0])

# Strategy by column type:
# - Numerical: median imputation (robust to outliers)
# - Categorical: mode or 'Unknown'
# - Text fields: 'None' or drop

# Numerical columns - median imputation
numerical_cols = df.select_dtypes(include=[np.number]).columns
for col in numerical_cols:
    if df[col].isnull().sum() > 0:
        median_val = df[col].median()
        df[col].fillna(median_val, inplace=True)
        print(f"  {col}: filled with median ({median_val:.2f})")

# Categorical columns with missing values
# holiday_name, festival_name, collector_notes - these are expected to be mostly null
df['holiday_name'].fillna('No Holiday', inplace=True)
df['festival_name'].fillna('No Festival', inplace=True)
df['collector_notes'].fillna('No Notes', inplace=True)

print("\nMissing values after:")
print(df.isnull().sum().sum(), "total missing values")


🔧 Handling Missing Values...

Missing values before:
holiday_name       104096
festival_name       99789
collector_notes    100564
dtype: int64

Missing values after:
0 total missing values


In [14]:
# ==============================================================================
# 5. ENCODE CATEGORICAL VARIABLES
# ==============================================================================

print("\n🏷️ Encoding Categorical Variables...")

# Identify categorical columns
categorical_cols = ['location_type', 'district', 'waste_type_primary', 'bin_material',
                    'road_accessibility', 'day_of_week', 'season', 'fill_level_category']

print(f"Categorical columns: {categorical_cols}")

# Strategy:
# - location_type: One-Hot (6 categories, important for model)
# - district: Target Encoding (16 categories, too many for one-hot)
# - waste_type_primary: One-Hot (6 categories)
# - Others: Label Encoding or One-Hot based on cardinality

# 5a. One-Hot Encoding for low-cardinality features
onehot_cols = ['location_type', 'waste_type_primary', 'bin_material', 'season']

df_encoded = df.copy()
for col in onehot_cols:
    dummies = pd.get_dummies(df[col], prefix=col, drop_first=True)
    df_encoded = pd.concat([df_encoded, dummies], axis=1)
    print(f"  ✅ One-Hot encoded '{col}' → {len(dummies.columns)} new columns")

# 5b. Label Encoding for ordinal features
ordinal_mappings = {
    'road_accessibility': {'poor': 0, 'limited': 1, 'good': 2, 'excellent': 3},
    'fill_level_category': {'low': 0, 'medium': 1, 'high': 2, 'critical': 3},
    'day_of_week': {'Monday': 0, 'Tuesday': 1, 'Wednesday': 2, 'Thursday': 3, 
                   'Friday': 4, 'Saturday': 5, 'Sunday': 6}
}

for col, mapping in ordinal_mappings.items():
    if col in df_encoded.columns:
        df_encoded[f'{col}_encoded'] = df_encoded[col].map(mapping)
        print(f"  ✅ Ordinal encoded '{col}'")

# 5c. Target Encoding for district (high cardinality)
district_means = df.groupby('district')['fill_level_percent'].mean()
df_encoded['district_encoded'] = df_encoded['district'].map(district_means)
print(f"  ✅ Target encoded 'district' (mean fill level)")

print(f"\nEncoded dataset shape: {df_encoded.shape}")


🏷️ Encoding Categorical Variables...
Categorical columns: ['location_type', 'district', 'waste_type_primary', 'bin_material', 'road_accessibility', 'day_of_week', 'season', 'fill_level_category']
  ✅ One-Hot encoded 'location_type' → 5 new columns
  ✅ One-Hot encoded 'waste_type_primary' → 5 new columns
  ✅ One-Hot encoded 'bin_material' → 2 new columns
  ✅ One-Hot encoded 'season' → 1 new columns
  ✅ Ordinal encoded 'road_accessibility'
  ✅ Ordinal encoded 'fill_level_category'
  ✅ Ordinal encoded 'day_of_week'
  ✅ Target encoded 'district' (mean fill level)

Encoded dataset shape: (107949, 75)


In [15]:
# ==============================================================================
# 6. SELECT FEATURES FOR MODELING
# ==============================================================================

print("\n🎯 Selecting Features for Modeling...")

# Features to EXCLUDE (leakage or not useful)
exclude_features = [
    # IDs and dates
    'collection_id', 'bin_id', 'collection_date', 'collection_time',
    # Text fields
    'holiday_name', 'festival_name', 'collector_notes', 'waste_composition_json',
    # Target leakage (only known after collection)
    'waste_weight_kg', 'overflow_reported', 'odor_complaint',
    # Original categorical columns (we have encoded versions)
    'location_type', 'district', 'waste_type_primary', 'bin_material', 
    'road_accessibility', 'day_of_week', 'season', 'fill_level_category',
    # Redundant
    'installation_date'
]

# Target variable
target = 'fill_level_percent'

# Feature columns
feature_cols = [col for col in df_encoded.columns if col not in exclude_features and col != target]

print(f"Target: {target}")
print(f"Number of features: {len(feature_cols)}")
print(f"\nFeature list:")
for i, col in enumerate(feature_cols, 1):
    print(f"  {i}. {col}")

# Create final datasets
X = df_encoded[feature_cols]
y = df_encoded[target]

print(f"\nX shape: {X.shape}")
print(f"y shape: {y.shape}")


🎯 Selecting Features for Modeling...
Target: fill_level_percent
Number of features: 54

Feature list:
  1. is_holiday
  2. is_festival_period
  3. days_since_last_collection
  4. temperature_c
  5. rainfall_mm
  6. humidity_percent
  7. capacity_liters
  8. has_lid
  9. nearby_population
  10. distance_to_depot_km
  11. latitude
  12. longitude
  13. month
  14. week_of_year
  15. is_weekend
  16. is_month_start
  17. is_month_end
  18. fill_rate_7day_avg
  19. fill_rate_7day_std
  20. weight_7day_avg
  21. prev_fill_level
  22. prev_days_since_collection
  23. bin_age_days
  24. fill_rate_per_day
  25. overflow_count_30day
  26. organic_percent
  27. recyclable_percent
  28. quarter
  29. day_of_month
  30. is_early_month
  31. is_late_month
  32. location_weekend
  33. fill_momentum
  34. capacity_fill_ratio
  35. weather_score
  36. population_fill_ratio
  37. overflow_risk_score
  38. location_type_hospitality
  39. location_type_industrial
  40. location_type_institutional
  41. 

In [16]:
import os

# Create directories if they don't exist
os.makedirs('C:/Users/user/Desktop/DecisionTree/data/processed/processed', exist_ok=True)

In [17]:
# ==============================================================================
# 7. SAVE PROCESSED DATA
# ==============================================================================

import os

# Create directories if they don't exist
os.makedirs('C:/Users/user/Desktop/DecisionTree/data/processed/processed', exist_ok=True)
print("✅ Created directory: ../data/processed")

# Save full encoded dataset
df_encoded.to_csv('C:/Users/user/Desktop/DecisionTree/data/processed/features_encoded.csv', index=False)
print("✅ Saved: C:/Users/user/Desktop/DecisionTree/data/processed/features_encoded.csv")

# Save feature matrix and target
X.to_csv('C:/Users/user/Desktop/DecisionTree/data/processed/X_features.csv', index=False)
y.to_csv('C:/Users/user/Desktop/DecisionTree/data/processed/y_target.csv', index=False)
print("✅ Saved: ../data/processed/X_features.csv")
print("✅ Saved: ../data/processed/y_target.csv")

# Save feature list
with open('C:/Users/user/Desktop/DecisionTree/data/processed/feature_list.txt', 'w') as f:
    for col in feature_cols:
        f.write(f"{col}\n")
print("✅ Saved: C:/Users/user/Desktop/DecisionTree/data/processed/feature_list.txt")

print("\n" + "=" * 60)
print("FEATURE ENGINEERING COMPLETE!")
print("=" * 60)
print(f"Final dataset: {X.shape[0]} rows, {X.shape[1]} features")

✅ Created directory: ../data/processed
✅ Saved: C:/Users/user/Desktop/DecisionTree/data/processed/features_encoded.csv
✅ Saved: ../data/processed/X_features.csv
✅ Saved: ../data/processed/y_target.csv
✅ Saved: C:/Users/user/Desktop/DecisionTree/data/processed/feature_list.txt

FEATURE ENGINEERING COMPLETE!
Final dataset: 107949 rows, 54 features
